## Subplots of Productive Months - scrapped, new idea

In [1]:
import numpy as np
from oneargopy.OneArgo import Argo
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.cm as cm # new package for the colorbar
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.ticker import LogLocator, FormatStrFormatter
from matplotlib.colors import LogNorm
import gsw
from tqdm import tqdm
import matplotlib.image as mpimg
import earthaccess
import h5netcdf
import seaborn as sns
import xarray as xr
from matplotlib.patches import Rectangle
import dask
import contourpy

from importlib import reload
import SO_tools as tools; reload(tools)

/opt/anaconda3/envs/IBIS_Project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'SO_tools' from '/Users/lilah/Documents/IBIS_Project/SO_tools.py'>

In [2]:
df_new = pd.read_csv('/Users/lilah/Documents/IBIS_Project/Data_Summer/newest_bgc_profiles_ext_alltime.csv')
# df_new = pd.read_csv('/Users/steviewalker/Documents/IBIS_Project/Data_Summer/new_bgc_profiles_BR_extended_area_all_time.csv') # for stevie

/var/folders/sr/nn_fjjln6s33md98tyfc3rd40000gn/T/ipykernel_12422/1697200288.py:1: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  df_new = pd.read_csv('/Users/lilah/Documents/IBIS_Project/Data_Summer/newest_bgc_profiles_ext_alltime.csv')


In [3]:
#convert to datetime
df_new['DATE'] = pd.to_datetime(df_new['DATE'],format = 'mixed')
df_new = df_new.dropna(subset=["DATE"]) #drop any cycles with no date or lat/lon 

#fix lat lon values with '--' (replace with nan) 
df_new['LATITUDE'] = pd.to_numeric(df_new['LATITUDE'], errors='coerce')
df_new['LONGITUDE'] = pd.to_numeric(df_new['LONGITUDE'], errors='coerce')

## Building function in words

group months by productive season (if month is 11, 12, 1, 2, 3, or 4)

adjust pressure 1x1 grid so can compare across profiles (previous code)

for one variable (input would be: chla, temp, salinity, stratification) calculate mean, median and standard deviation for each month in productive season

6 subplots for nov-apr with 200 dbar and line showing average (mean?) variable and standard deviation (fillbetween shading) across years

In [4]:
cond1 = (df_new['month'] >= 11)
cond2 = (df_new['month'] <= 4)

df_m = df_new[cond1 | cond2]
df_m

,Unnamed: 0.1,Unnamed: 0,WMOID,CYCLE_NUMBER,DIRECTION,DATE,DATE_QC,LATITUDE,LONGITUDE,POSITION_QC,...,DOWNWELLING_PAR_ADJUSTED_ERROR,year,month,season,SA,CT,rho,sigma0,nsq,MLP
73,73,73,1901153,292,A,2019-01-10 03:00:33.999977+00:00,1,-59.3080,167.9760,1,...,NaN,2019,1,2018-2019,34.091632,5.944086,1026.741598,26.717176,-2.198041e-06,61.20
74,74,74,1901153,292,A,2019-01-10 03:00:33.999977+00:00,1,-59.3080,167.9760,1,...,NaN,2019,1,2018-2019,34.090759,5.948644,1026.765685,26.715923,3.912762e-06,61.20
75,75,75,1901153,292,A,2019-01-10 03:00:33.999977+00:00,1,-59.3080,167.9760,1,...,NaN,2019,1,2018-2019,34.092704,5.926741,1026.818317,26.720176,-1.051022e-07,61.20
76,76,76,1901153,292,A,2019-01-10 03:00:33.999977+00:00,1,-59.3080,167.9760,1,...,NaN,2019,1,2018-2019,34.089905,5.909985,1026.861973,26.720061,1.070486e-05,61.20
77,77,77,1901153,292,A,2019-01-10 03:00:33.999977+00:00,1,-59.3080,167.9760,1,...,NaN,2019,1,2018-2019,34.093811,5.841052,1026.921997,26.731659,3.793829e-06,61.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1546185,1546185,1547165,7902379,7,A,2026-04-22 01:17:48.000132+00:00,1,-58.2467,158.3533,1,...,NaN,2026,4,2025-2026,34.913753,1.548936,1035.902787,27.806283,1.835100e+02,98.34
1546186,1546186,1547166,7902379,7,A,2026-04-22 01:17:48.000132+00:00,1,-58.2467,158.3533,1,...,NaN,2026,4,2025-2026,34.913819,1.525888,1036.133826,27.808038,1.855100e+02,98.34
1546187,1546187,1547167,7902379,7,A,2026-04-22 01:17:48.000132+00:00,1,-58.2467,158.3533,1,...,NaN,2026,4,2025-2026,34.912878,1.477915,1036.361383,27.810809,1.875100e+02,98.34
1546188,1546188,1547168,7902379,7,A,2026-04-22 01:17:48.000132+00:00,1,-58.2467,158.3533,1,...,NaN,2026,4,2025-2026,34.911938,1.444398,1036.592989,27.812502,1.895100e+02,98.34


In [5]:
cmean = df_m['CHLA_ADJUSTED'].mean()
cmedian = df_m['CHLA_ADJUSTED'].median()
cstd = df_m['CHLA_ADJUSTED'].std()

In [6]:
print(df_new['DATE'].dtypes)

datetime64[us, UTC]


## regridding data

In [17]:
time_values = pd.to_datetime(df_new['DATE'], format='mixed').values
pres_values = df_new['PRES'].values
param_values = df_new['CHLA_ADJUSTED'].values
# Remove NaN values
valid_indices = ~np.isnan(time_values) & ~np.isnan(pres_values) & ~np.isnan(param_values)
time_values = time_values[valid_indices]
pres_values = pres_values[valid_indices]
param_values = param_values[valid_indices]
# Convert time_values to float because it makes gridding data easier
time_values_num = mdates.date2num(time_values)
# Unique values for creating grids
unique_times_num = np.unique(time_values_num)
# Create a pressure axis with regular intervals, covering all existing values
intp_pres = np.arange(np.ceil(min(pres_values)), np.floor(max(pres_values)))
# Create grid for interpolation
time_grid, pres_grid = np.meshgrid(unique_times_num, intp_pres)
# Set param_gridded to NaN array with the same shape as the grid
param_gridded = np.full(time_grid.shape, np.nan)
# Create a DataFrame
d_f = pd.DataFrame({
    'time': time_values_num,
    'pressure': pres_values,
    'param': param_values
})
# Pivot the DataFrame to create a grid
param_gridded_df = d_f.pivot_table(
    index='pressure',
    columns='time',
    values='param',
    aggfunc='first'
)
# Create a new index that contains original and regularly spaced pressure values
all_pres = np.sort(np.unique(np.concatenate([pres_values, intp_pres])))
# Reindex the DataFrame to the combined depth axis
param_gridded_df = param_gridded_df.reindex(index=all_pres, columns=unique_times_num)
# Perform linear interpolation to the new depth axis without extrapolation
param_gridded_df.interpolate(method='linear', limit_area='inside', axis=0, inplace=True)
# Extract the values to the regularly spaced depth values
param_gridded_df = param_gridded_df.reindex(index=intp_pres, columns=unique_times_num)
# Assigning data to variable to graph
param_gridded = param_gridded_df.values

print(param_gridded)

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


## FUNCTION

In [15]:
def interannual_cycles(df, month, variable, ax_position, color, title, title_size):

    
    # regridding data (from oneargopy code) ---------------
    
    time_values = pd.to_datetime(df['DATE'], format='mixed').values
    pres_values = df['PRES'].values
    param_values = df[variable].values
    # Remove NaN values
    valid_indices = ~np.isnan(time_values) & ~np.isnan(pres_values) & ~np.isnan(param_values)
    time_values = time_values[valid_indices]
    pres_values = pres_values[valid_indices]
    param_values = param_values[valid_indices]
    # Convert time_values to float because it makes gridding data easier
    time_values_num = mdates.date2num(time_values)
    # Unique values for creating grids
    unique_times_num = np.unique(time_values_num)
    # Create a pressure axis with regular intervals, covering all existing values
    intp_pres = np.arange(np.ceil(min(pres_values)), np.floor(max(pres_values)))
    # Create grid for interpolation
    time_grid, pres_grid = np.meshgrid(unique_times_num, intp_pres)
    # Set param_gridded to NaN array with the same shape as the grid
    param_gridded = np.full(time_grid.shape, np.nan)
    # Create a DataFrame
    d_f = pd.DataFrame({
        'time': time_values_num,
        'pressure': pres_values,
        'param': param_values
    })
    # Pivot the DataFrame to create a grid
    param_gridded_df = d_f.pivot_table(
        index='pressure',
        columns='time',
        values='param',
        aggfunc='first'
    )
    # Create a new index that contains original and regularly spaced pressure values
    all_pres = np.sort(np.unique(np.concatenate([pres_values, intp_pres])))
    # Reindex the DataFrame to the combined depth axis
    param_gridded_df = param_gridded_df.reindex(index=all_pres, columns=unique_times_num)
    # Perform linear interpolation to the new depth axis without extrapolation
    param_gridded_df.interpolate(method='linear', limit_area='inside', axis=0, inplace=True)
    # Extract the values to the regularly spaced depth values
    param_gridded_df = param_gridded_df.reindex(index=intp_pres, columns=unique_times_num)
    # Assigning data to variable to graph
    param_gridded = param_gridded_df.values

    pressure = pres_grid.mean()

    mean_v = df[param_gridded].mean()
    std_v = param_gridded.std()

    fig = plt.figure(figsize=(10,12), layout='constrained')
    ax = fig.add_subplot(ax_position)
    plot = ax.plot(mean_v, pressure, c=color)

    ax.set_ylim(0,200)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=title_size)

interannual_cycles(df=df_m, month = 11, variable = 'CHLA_ADJUSTED', ax_position = 231, color = 'k', title = 'Nov', title_size=12)

KeyError: "None of [Index([                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  (nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan